In [99]:
'''!pip install -q requirements.txt'''

'!pip install -q requirements.txt'

In [100]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [101]:
import yfinance as yf
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import gym_anytrading
import stable_baselines3
from gym_anytrading.envs import Actions
from stable_baselines3 import PPO

In [102]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversions

Author: Matheus dos Anjos

gym_anytrading   : 2.0.0
gymnasium        : 1.2.3
matplotlib       : 3.10.6
numpy            : 2.3.5
pandas           : 2.3.3
stable_baselines3: 2.7.1
yfinance         : 1.0



In [103]:
def get_stock_data(ticker):
    start_date = '2016-01-01'
    end_date = '2025-12-25'

    dados = yf.download(ticker, start = start_date, end= end_date)

    dados = dados[['Close']]

    return dados

In [104]:
df = get_stock_data('BTC-USD')

[*********************100%***********************]  1 of 1 completed


In [105]:
df.shape

(3646, 1)

In [106]:
df.head(3)

Price,Close
Ticker,BTC-USD
Date,
2016-01-01,434.334015
2016-01-02,433.437988
2016-01-03,430.010986


In [107]:
df.tail(3)

Price,Close
Ticker,BTC-USD
Date,
2025-12-22,88490.015625
2025-12-23,87414.000000
2025-12-24,87611.960938


In [108]:
window_size = 10
start_index = window_size
end_index = len(df)

In [109]:
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

df = df[['Close']].astype(float)
df = df.dropna()

end_index = len(df)

In [110]:
#Forma de importação antiga

'''env = gym.make('stocks-v0',
    df=df,
    window_size=window_size,
    frame_bound=(start_index, end_index), 
)

print("Ambiente criado com sucesso!")'''

'env = gym.make(\'stocks-v0\',\n    df=df,\n    window_size=window_size,\n    frame_bound=(start_index, end_index), \n)\n\nprint("Ambiente criado com sucesso!")'

In [111]:
from gym_anytrading.envs import StocksEnv

class CustomStocksEnv(StocksEnv):
    
    def _process_data(self):
        
        prices = self.df['Close'].to_numpy()
        
        start = self.frame_bound[0] - self.window_size
        end   = self.frame_bound[1]
        prices = prices[start:end]

        diff = np.insert(np.diff(prices, axis=0), 0, 0)

        signal_features = np.column_stack((prices, diff))
        
        return prices.astype(np.float32), signal_features.astype(np.float32)

In [112]:
env = CustomStocksEnv(df = df,
                         window_size = window_size,
                         frame_bound = (start_index, end_index))

In [113]:
saldo = 100000
historico_saldo = ['saldo']
num_acoes_manter = 0
status_decisao = {Actions.Sell: 0, Actions.Buy: 0}

In [114]:
observation, info = env.reset(seed = 42)

In [115]:
modelo_trading = PPO('MlpPolicy', env, seed = 42, verbose = 1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


AttributeError: module 'torch' has no attribute '_utils'

In [116]:
modelo_trading.learn(total_timesteps = 10000)

NameError: name 'modelo_trading' is not defined

In [ ]:
step = 0

while True:
    action, _states = modelo_trading.predict(observation)
    current_price = env.unwrapped.prices[env.unwrapped._current_tick]
    observation, neward, terminated, truncated, info = env.step(action)
    trade_amount = saldo * 0.10

    if action == Actions.Buy.value:
        
        shares_to_buy = trade_amount / current_price  
        
        num_acoes_manter += shares_to_buy
        saldo -= trade_amount
        print(f'{step}: Comprar {shares_to_buy:.2f} ações por ${current_price:.2f} cada uma | Saldo: ${saldo:.2f}')

    elif action == Actions.Sell.value and num_acoes_manter > 0:
        saldo += num_acoes_manter * current_price
        print(f'{step}: Vender {num_acoes_manter:.2f} ações por ${current_price:.2f} cada uma | Saldo ${saldo:.2f}')

        num_acoes_manter = 0

    else:
        print(f'{step}: Manter | Valor corrente da ação: ${current_price:.2f} | Saldo: ${saldo:.2f}')

    status_decisao[Actions(action)] += 1
    historico_saldo.append(saldo)

    step += 1

    if terminated or truncated:
        break

0: Manter | Valor corrente da ação: $62504.79 | Saldo: $100000.00


NameError: name 'shares_to_buy' is not defined

In [ ]:
if num_acoes_manter > 0:
    saldo += num_acoes_manter * current_price
    print(f'Venda Final de {num_acoes_manter:.2f} ações por ${current_price:.2f} cada uma | Saldo: ${saldo:.2f}')